In [5]:
# ============================================================
# PHASE 1: DATA EXPLORATION & UNIFIED CORPUS CREATION
# ============================================================
# Goal: Load CSV + PDFs, compare them, create unified dataset
# ============================================================

# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import pdfplumber
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Define paths (adjust these to match YOUR folder structure)
# These are relative paths from the notebooks/ folder
BASE_DIR = Path('../data/raw/archive')
CSV_PATH = BASE_DIR / 'Resume' / 'resume.csv'
PDF_DIR = BASE_DIR / 'data'

print("=" * 60)
print("PHASE 1: DATA EXPLORATION")
print("=" * 60)
print(f"\nBase directory: {BASE_DIR}")
print(f"CSV path: {CSV_PATH}")
print(f"PDF directory: {PDF_DIR}")

PHASE 1: DATA EXPLORATION

Base directory: ..\data\raw\archive
CSV path: ..\data\raw\archive\Resume\resume.csv
PDF directory: ..\data\raw\archive\data


In [6]:
# ============================================================
# STEP 1: LOAD CSV DATA
# ============================================================
print("\n" + "=" * 60)
print("LOADING CSV DATA")
print("=" * 60)

# Load the CSV file
df_csv = pd.read_csv(CSV_PATH)

print(f"\nCSV Shape: {df_csv.shape}")
print(f"  Rows: {df_csv.shape[0]}")
print(f"  Columns: {df_csv.shape[1]}")

print(f"\nColumn Names:")
for i, col in enumerate(df_csv.columns):
    print(f"  {i+1}. {col}")

print(f"\nFirst 3 rows:")
print(df_csv.head(3))

print(f"\nData Types:")
print(df_csv.dtypes)

print(f"\nMissing Values:")
print(df_csv.isnull().sum())


LOADING CSV DATA

CSV Shape: (2484, 4)
  Rows: 2484
  Columns: 4

Column Names:
  1. ID
  2. Resume_str
  3. Resume_html
  4. Category

First 3 rows:
         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  

Data Types:
ID             int64
Resume_str       str
Resume_html      str
Category         str
dtype: object

Missing Values:
ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64


In [9]:
# ============================================================
# STEP 2: EXPLORE PDF FILES (CORRECTED)
# ============================================================
print("\n" + "=" * 60)
print("EXPLORING PDF FILES")
print("=" * 60)

# The actual professions are one level deeper
PROFESSION_DIR = PDF_DIR / 'data'

# Count total PDFs
pdf_files = list(PROFESSION_DIR.rglob('*.pdf'))
print(f"\nTotal PDFs found: {len(pdf_files)}")

# Show PDFs per profession
print(f"\nPDFs per profession:")
profession_count = {}
for profession_folder in PROFESSION_DIR.iterdir():
    if profession_folder.is_dir():
        pdfs = list(profession_folder.glob('*.pdf'))
        profession_count[profession_folder.name] = len(pdfs)

for prof, count in sorted(profession_count.items(), key=lambda x: x[1], reverse=True):
    print(f"  {prof}: {count} PDFs")

print(f"\nTotal professions: {len(profession_count)}")


EXPLORING PDF FILES

Total PDFs found: 2484

PDFs per profession:
  BUSINESS-DEVELOPMENT: 120 PDFs
  INFORMATION-TECHNOLOGY: 120 PDFs
  ACCOUNTANT: 118 PDFs
  ADVOCATE: 118 PDFs
  CHEF: 118 PDFs
  ENGINEERING: 118 PDFs
  FINANCE: 118 PDFs
  AVIATION: 117 PDFs
  FITNESS: 117 PDFs
  SALES: 116 PDFs
  BANKING: 115 PDFs
  CONSULTANT: 115 PDFs
  HEALTHCARE: 115 PDFs
  CONSTRUCTION: 112 PDFs
  PUBLIC-RELATIONS: 111 PDFs
  HR: 110 PDFs
  DESIGNER: 107 PDFs
  ARTS: 103 PDFs
  TEACHER: 102 PDFs
  APPAREL: 97 PDFs
  DIGITAL-MEDIA: 96 PDFs
  AGRICULTURE: 63 PDFs
  AUTOMOBILE: 36 PDFs
  BPO: 22 PDFs

Total professions: 24


In [10]:
# ============================================================
# STEP 3: EXTRACT TEXT FROM SAMPLE PDFs
# ============================================================
print("\n" + "=" * 60)
print("EXTRACTING TEXT FROM SAMPLE PDFs")
print("=" * 60)

PROFESSION_DIR = PDF_DIR / 'data'

# Function to extract text from a single PDF
def extract_text_from_pdf(pdf_path):
    """
    Extract text from a PDF file using pdfplumber
    
    Args:
        pdf_path: Path to PDF file
        
    Returns:
        Extracted text as string, or None if extraction fails
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = ""
            for page in pdf.pages:
                text += page.extract_text() + "\n"
        return text.strip()
    except Exception as e:
        return None

# Test on first 3 PDFs
print("\nTesting PDF extraction on 3 sample PDFs:\n")

sample_count = 0
for profession_folder in PROFESSION_DIR.iterdir():
    if profession_folder.is_dir() and sample_count < 3:
        pdfs = list(profession_folder.glob('*.pdf'))
        if pdfs:
            pdf_path = pdfs[0]
            extracted_text = extract_text_from_pdf(pdf_path)
            
            print(f"Profession: {profession_folder.name}")
            print(f"PDF: {pdf_path.name}")
            print(f"Extraction success: {extracted_text is not None}")
            if extracted_text:
                print(f"Text length: {len(extracted_text)} characters")
                print(f"First 200 characters:\n{extracted_text[:200]}\n")
            print("-" * 60 + "\n")
            
            sample_count += 1


EXTRACTING TEXT FROM SAMPLE PDFs

Testing PDF extraction on 3 sample PDFs:

Profession: ACCOUNTANT
PDF: 10554236.pdf
Extraction success: True
Text length: 24153 characters
First 200 characters:
ACCOUNTANT
Summary
Financial Accountant specializing in financial planning, reporting and analysis within the Department of Defense.
Highlights
Account reconciliations
Results-oriented Accounting oper

------------------------------------------------------------

Profession: ADVOCATE
PDF: 10186968.pdf
Extraction success: True
Text length: 3917 characters
First 200 characters:
CHILD FAMILY ADVOCATE
Professional Profile
Talented administrative professional with background in accounting and finance. Extensive knowledge of AR/AP, Microsoft Excel and Quick
Books-software skills

------------------------------------------------------------

Profession: AGRICULTURE
PDF: 10953078.pdf
Extraction success: True
Text length: 5263 characters
First 200 characters:
RN STAFF NURSE
Professional Experience
RN St

In [11]:
# ============================================================
# STEP 4: EXTRACT TEXT FROM ALL PDFs & CREATE COMPARISON
# ============================================================
print("\n" + "=" * 60)
print("EXTRACTING ALL PDFs & COMPARING WITH CSV")
print("=" * 60)

PROFESSION_DIR = PDF_DIR / 'data'

# Dictionary to store PDF extraction results
pdf_extraction = {}

print(f"\nExtracting text from {len(pdf_files)} PDFs...")
# print("(This may take a few minutes)\n")

# Extract text from all PDFs
for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    pdf_id = pdf_path.stem  # Get filename without extension
    extracted_text = extract_text_from_pdf(pdf_path)
    pdf_extraction[pdf_id] = extracted_text

print(f"\n✓ Successfully extracted: {len(pdf_extraction)} PDFs")
print(f"✗ Failed extractions: {len(pdf_files) - len(pdf_extraction)}")

# Now compare with CSV
print("\n" + "=" * 60)
print("COMPARING PDF vs CSV TEXT")
print("=" * 60)

csv_ids = set(df_csv['ID'].astype(str))
pdf_ids = set(pdf_extraction.keys())

print(f"\nIDs in CSV: {len(csv_ids)}")
print(f"IDs in PDFs: {len(pdf_ids)}")
print(f"IDs in both: {len(csv_ids & pdf_ids)}")
print(f"IDs only in CSV: {len(csv_ids - pdf_ids)}")
print(f"IDs only in PDFs: {len(pdf_ids - csv_ids)}")

# Show examples of IDs that don't match
if csv_ids - pdf_ids:
    print(f"\nExamples of IDs in CSV but NOT in PDFs:")
    print(f"  {list(csv_ids - pdf_ids)[:5]}")

if pdf_ids - csv_ids:
    print(f"\nExamples of IDs in PDFs but NOT in CSV:")
    print(f"  {list(pdf_ids - csv_ids)[:5]}")


EXTRACTING ALL PDFs & COMPARING WITH CSV

Extracting text from 2484 PDFs...
(This may take a few minutes)



Extracting PDFs: 100%|██████████| 2484/2484 [13:03<00:00,  3.17it/s]


✓ Successfully extracted: 2484 PDFs
✗ Failed extractions: 0

COMPARING PDF vs CSV TEXT

IDs in CSV: 2484
IDs in PDFs: 2484
IDs in both: 2484
IDs only in CSV: 0
IDs only in PDFs: 0


In [12]:
# ============================================================
# STEP 5: TEXT QUALITY COMPARISON (PDF vs CSV)
# ============================================================
print("\n" + "=" * 60)
print("COMPARING TEXT QUALITY: PDF EXTRACTION vs CSV")
print("=" * 60)

# Create a comparison dataframe
comparison_data = []

print("\nAnalyzing text differences...")

for idx, row in tqdm(df_csv.iterrows(), total=len(df_csv), desc="Comparing texts"):
    csv_id = str(row['ID'])
    csv_text = row['Resume_str']
    pdf_text = pdf_extraction.get(csv_id, "")
    
    # Check if texts exist and compare
    csv_exists = bool(csv_text and len(str(csv_text).strip()) > 0)
    pdf_exists = bool(pdf_text and len(pdf_text.strip()) > 0)
    
    # Determine match status
    if not csv_exists and not pdf_exists:
        match_status = "BOTH_EMPTY"
    elif not csv_exists:
        match_status = "CSV_EMPTY"
    elif not pdf_exists:
        match_status = "PDF_EMPTY"
    else:
        # Check if texts are similar (using length comparison as proxy)
        csv_len = len(str(csv_text).strip())
        pdf_len = len(pdf_text.strip())
        diff_ratio = abs(csv_len - pdf_len) / max(csv_len, pdf_len)
        
        if diff_ratio < 0.1:  # Within 10% difference
            match_status = "SIMILAR"
        elif diff_ratio < 0.3:  # Within 30% difference
            match_status = "SLIGHTLY_DIFFERENT"
        else:
            match_status = "VERY_DIFFERENT"
    
    comparison_data.append({
        'ID': csv_id,
        'Category': row['Category'],
        'CSV_length': len(str(csv_text).strip()) if csv_exists else 0,
        'PDF_length': len(pdf_text.strip()) if pdf_exists else 0,
        'Match_status': match_status
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 60)
print("COMPARISON RESULTS")
print("=" * 60)

print(f"\nMatch Status Distribution:")
print(df_comparison['Match_status'].value_counts())

print(f"\nAverage text lengths:")
print(f"  CSV: {df_comparison['CSV_length'].mean():.0f} characters")
print(f"  PDF: {df_comparison['PDF_length'].mean():.0f} characters")

print(f"\nSample comparisons:")
print(df_comparison.head(10))


COMPARING TEXT QUALITY: PDF EXTRACTION vs CSV

Analyzing text differences...


Comparing texts: 100%|██████████| 2484/2484 [00:00<00:00, 40601.74it/s]


COMPARISON RESULTS

Match Status Distribution:
Match_status
SIMILAR               2311
SLIGHTLY_DIFFERENT     170
VERY_DIFFERENT           2
BOTH_EMPTY               1
Name: count, dtype: int64

Average text lengths:
  CSV: 6282 characters
  PDF: 5934 characters

Sample comparisons:
         ID Category  CSV_length  PDF_length Match_status
0  16852973       HR        5429        5047      SIMILAR
1  22323967       HR        5560        5232      SIMILAR
2  33176873       HR        7706        7348      SIMILAR
3  27018550       HR        2839        2664      SIMILAR
4  17812897       HR        9160        8895      SIMILAR
5  11592605       HR        5468        5163      SIMILAR
6  25824789       HR        5238        5014      SIMILAR
7  15375009       HR        9020        8653      SIMILAR
8  11847784       HR        6700        6386      SIMILAR
9  32896934       HR        5966        5618      SIMILAR


In [13]:
# ============================================================
# STEP 6: CREATE UNIFIED CORPUS
# ============================================================
print("\n" + "=" * 60)
print("CREATING UNIFIED CORPUS")
print("=" * 60)

# Create unified dataframe with both sources
unified_data = []

for idx, row in tqdm(df_csv.iterrows(), total=len(df_csv), desc="Creating unified corpus"):
    csv_id = str(row['ID'])
    csv_text = str(row['Resume_str']).strip() if pd.notna(row['Resume_str']) else ""
    pdf_text = pdf_extraction.get(csv_id, "").strip()
    category = row['Category']
    
    # Decide which text to use (prefer CSV since it's already clean)
    final_text = csv_text if csv_text else pdf_text
    
    # Determine source
    if csv_text and pdf_text:
        source = "CSV_and_PDF"
    elif csv_text:
        source = "CSV_only"
    elif pdf_text:
        source = "PDF_only"
    else:
        source = "NONE"
    
    unified_data.append({
        'ID': csv_id,
        'Resume_text': final_text,
        'Resume_length': len(final_text),
        'Category': category,
        'Source': source,
        'CSV_available': bool(csv_text),
        'PDF_available': bool(pdf_text)
    })

df_unified = pd.DataFrame(unified_data)

print(f"\n✓ Unified corpus created!")
print(f"  Total resumes: {len(df_unified)}")
print(f"  Total text: {df_unified['Resume_length'].sum():,} characters")

print(f"\nSource Distribution:")
print(df_unified['Source'].value_counts())

print(f"\nResume length statistics:")
print(df_unified['Resume_length'].describe())

print(f"\nCategory distribution:")
print(df_unified['Category'].value_counts())

print(f"\nFirst 5 rows of unified corpus:")
print(df_unified.head())


CREATING UNIFIED CORPUS


Creating unified corpus: 100%|██████████| 2484/2484 [00:00<00:00, 33544.60it/s]


✓ Unified corpus created!
  Total resumes: 2484
  Total text: 15,603,485 characters

Source Distribution:
Source
CSV_and_PDF    2483
NONE              1
Name: count, dtype: int64

Resume length statistics:
count     2484.000000
mean      6281.596216
std       2769.253919
min          0.000000
25%       5146.750000
50%       5874.000000
75%       7215.000000
max      38813.000000
Name: Resume_length, dtype: float64

Category distribution:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER 

In [14]:
# ============================================================
# STEP 7: SAVE UNIFIED CORPUS
# ============================================================
print("\n" + "=" * 60)
print("SAVING UNIFIED CORPUS")
print("=" * 60)

# Create output directory if it doesn't exist
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Save unified corpus
output_path = output_dir / 'resumes_unified.csv'
df_unified.to_csv(output_path, index=False)

print(f"\n✓ Saved to: {output_path}")
print(f"  File size: {output_path.stat().st_size / (1024*1024):.2f} MB")

# Also save a summary report
print("\n" + "=" * 60)
print("PHASE 1 SUMMARY REPORT")
print("=" * 60)

summary = f"""
PHASE 1: DATA EXPLORATION & UNIFIED CORPUS CREATION
{'='*60}

DATA SOURCES:
- CSV file: {CSV_PATH}
- PDF directory: {PROFESSION_DIR}

STATISTICS:
- Total resumes: {len(df_unified)}
- Total characters: {df_unified['Resume_length'].sum():,}
- Average resume length: {df_unified['Resume_length'].mean():.0f} characters
- Min length: {df_unified['Resume_length'].min():.0f} characters
- Max length: {df_unified['Resume_length'].max():.0f} characters

SOURCE QUALITY:
- CSV + PDF available: {(df_unified['Source'] == 'CSV_and_PDF').sum()} ({(df_unified['Source'] == 'CSV_and_PDF').sum()/len(df_unified)*100:.1f}%)
- CSV only: {(df_unified['Source'] == 'CSV_only').sum()}
- PDF only: {(df_unified['Source'] == 'PDF_only').sum()}
- No text: {(df_unified['Source'] == 'NONE').sum()}

PROFESSIONS: {len(df_unified['Category'].unique())}
{'-'*60}
"""

for category in sorted(df_unified['Category'].unique()):
    count = (df_unified['Category'] == category).sum()
    summary += f"{category:30s}: {count:3d} resumes\n"

summary += f"""
{'='*60}

DELIVERABLES:
✓ resumes_unified.csv - Unified corpus with all resume text
✓ Phase 1 completed successfully!

NEXT STEPS:
→ Phase 2: NER Training & Labeling
  - Sample 500-1000 resumes for labeling
  - Label entities (SKILL, JOB_TITLE, COMPANY, etc.)
  - Train spaCy NER model

"""

print(summary)

# Save summary to file
summary_path = output_dir / 'phase1_summary.txt'
with open(summary_path, 'w') as f:
    f.write(summary)

print(f"\n✓ Summary saved to: {summary_path}")


SAVING UNIFIED CORPUS

✓ Saved to: ..\data\processed\resumes_unified.csv
  File size: 15.03 MB

PHASE 1 SUMMARY REPORT

PHASE 1: DATA EXPLORATION & UNIFIED CORPUS CREATION

DATA SOURCES:
- CSV file: ..\data\raw\archive\Resume\resume.csv
- PDF directory: ..\data\raw\archive\data\data

STATISTICS:
- Total resumes: 2484
- Total characters: 15,603,485
- Average resume length: 6282 characters
- Min length: 0 characters
- Max length: 38813 characters

SOURCE QUALITY:
- CSV + PDF available: 2483 (100.0%)
- CSV only: 0
- PDF only: 0
- No text: 1

PROFESSIONS: 24
------------------------------------------------------------
ACCOUNTANT                    : 118 resumes
ADVOCATE                      : 118 resumes
AGRICULTURE                   :  63 resumes
APPAREL                       :  97 resumes
ARTS                          : 103 resumes
AUTOMOBILE                    :  36 resumes
AVIATION                      : 117 resumes
BANKING                       : 115 resumes
BPO                      

UnicodeEncodeError: 'charmap' codec can't encode character '\u2713' in position 1748: character maps to <undefined>